In [0]:
CATALOG = "dbr_dev"
# dbutils.widgets.get("catalog")

GOLD = f"{CATALOG}.skybound_gold"

In [0]:
spark.sql(f"""
  CREATE OR REPLACE TABLE {GOLD}.agg_country_current AS

    WITH latest_obs AS (
        SELECT *
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (
                    PARTITION BY station_id
                    ORDER BY obs_time DESC
                ) AS rn
            FROM {GOLD}.fact_airport_weather_reports
        )
        WHERE rn = 1
    )
  

  SELECT
    f.airport_country,
    COUNT(*)                       AS airport_count,
    ROUND(AVG(f.temp_c), 1)        AS avg_temp_c,
    ROUND(AVG(f.wind_speed_kt), 1) AS avg_wind_kt,
    ROUND(AVG(f.visibility_sm), 2) AS avg_visibility_sm,

    -- Flight category breakdown
    ROUND(AVG(CASE WHEN f.flight_category = 'VFR' THEN 100.0 ELSE 0 END), 1) AS pct_vfr,
    ROUND(AVG(CASE WHEN f.flight_category = 'MVFR' THEN 100.0 ELSE 0 END), 1) AS pct_mvfr,
    ROUND(AVG(CASE WHEN f.flight_category = 'IFR' THEN 100.0 ELSE 0 END), 1) AS pct_ifr,
    ROUND(AVG(CASE WHEN f.flight_category = 'LIFR' THEN 100.0 ELSE 0 END), 1) AS pct_lifr,

    -- Hazard counts
    SUM(CAST(f.has_precipitation AS INT)) AS airports_with_precip,
    SUM(CAST(f.has_freezing      AS INT)) AS airports_with_freezing,
    SUM(CAST(f.has_thunderstorm  AS INT)) AS airports_with_thunderstorm,
    SUM(CAST(f.has_fog           AS INT)) AS airports_with_fog,

    MAX(f.obs_time) AS as_of_time

  FROM latest_obs f
  GROUP BY f.airport_country
""")